# 🔧 How Gemini *Selects*, *References*, and *Calls* Your Python Tools

### Dinesh AI Academy | Day 4 — Agents & MCP (Deep Dive)

**Learning objective:**
By the end of this notebook you will be able to draw, from memory, the exact
sequence of events that happens between "Gemini decides to use a tool" and
"a line of your Python code actually runs" — and say precisely **who** does
**what**, and **on which machine**, at every single step.

**Why this notebook exists:** the [Building AI Agents](./1-Building_AI_Agents.ipynb)
notebook *used* function calling to build an agent loop. This notebook opens
the hood and looks at the machinery itself — how a plain Python function
becomes something Gemini can "see", how Gemini decides to use it, and where
the actual function execution happens. Once this is crystal clear, agents,
MCP, and every framework you'll ever use (LangChain, ADK, OpenAI Agents SDK)
stop looking like magic.

## 1. The One Question That Fixes 90% of the Confusion

> ### ❓ "Does Gemini run my Python code?"
> ### 🚫 **No. Never. Not once. Not even a little bit.**

Gemini is a language model running on Google's servers. It has never seen
your laptop, your Python interpreter, or your function's source code. **All
it can ever do is generate text** — and "calling a tool" is really Gemini
generating a very specific, structured *piece of text* that says, in effect:

> *"If I were you, I'd now call the function named `get_weather` with the
> argument `city='Tokyo'`."*

That's it. That's the entire "call". Gemini's job ends the moment it produces
that structured text. **Your own code** is the thing that reads that text,
finds the real Python function, and runs it.

```text
   YOUR COMPUTER                                    GOOGLE'S SERVERS
   ───────────────                                   ─────────────────
   def get_weather(city):                              Gemini model
       ...real code...                                 (has NEVER seen the
        │                                                code above — only
        │  1. send: function NAME +                     its name + schema)
        │     SCHEMA (no source code)
        ├──────────────────────────────────────────────▶
        │                                               2. reads prompt + schema
        │                                               3. decides: "I should
        │                                                   request get_weather"
        │  4. returns: plain TEXT that says
        │     {"name": "get_weather",
        │      "args": {"city": "Tokyo"}}
        ◀──────────────────────────────────────────────┤
        │
        │  5. YOUR code reads that text,
        │     finds the REAL function, and
        │     calls it:  get_weather("Tokyo")
        │     ← this is the ONLY line in this
        │       entire diagram that executes code
        │
        │  6. send the RESULT back
        ├──────────────────────────────────────────────▶
        │                                               7. reads the result,
        │  8. returns: final answer text                   writes an answer
        ◀──────────────────────────────────────────────┤
```

Keep this diagram in your head for the rest of the notebook — every section
below is just zooming into one arrow of it.

## 2. Setup — Gemini API Key

Same pattern as every other notebook this week: works locally (via `.env`)
or in Google Colab (via Secrets), no code changes needed.

**Never publish your API key in a notebook, GitHub repository, Moodle,
WhatsApp group, or screenshot.**

In [1]:
# Install the current Google GenAI Python SDK.
# In Google Colab, run this cell once.

!pip -q install -U google-genai

In [ ]:
from google import genai
from google.genai import types
import inspect
import json
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

client = genai.Client(api_key=GAISTUDIO_API_KEY)

# You can change this model if your account has access to another Gemini model.
MODEL = "gemini-3.5-flash-lite"

print("Gemini client is ready. Model:", MODEL)

## 3. What Gemini Actually Receives About a Function (Hint: Not the Function)

Here is one real Python function, exactly as you'd write it for any normal
program — nothing "AI" about it yet.

In [ ]:
def get_weather(city: str) -> dict:
    """Get the current simulated weather for a city.

    Args:
        city: Name of the city, e.g. "Tokyo".
    """
    WEATHER_DB = {
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    }
    data = WEATHER_DB.get(city.strip().lower(), {"temp_c": 20, "condition": "unknown"})
    return {"city": city, **data}

# Sanity check — plain Python, no Gemini involved.
print(get_weather("Tokyo"))

{'city': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}


To let Gemini *reference* this function as a tool, we don't send Gemini the
function. We hand-write a small JSON object — a **declaration** — that
describes its *interface*: a name, a plain-English description, and the
shape of its arguments. This is the exact same pattern used in the
[Building AI Agents](./1-Building_AI_Agents.ipynb) notebook.

In [ ]:
weather_declaration = {
    "name": "get_weather",
    "description": "Gets the current simulated weather (temperature in Celsius, condition) for a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'Tokyo'."}
        },
        "required": ["city"]
    }
}

# THIS -- and only this -- is what crosses the network to Google's servers.
print(json.dumps(weather_declaration, indent=2))

{
  "name": "get_weather",
  "description": "Gets the current simulated weather (temperature in Celsius, condition) for a city.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. 'Tokyo'."
      }
    },
    "required": [
      "city"
    ]
  }
}


> **Read that JSON out loud in class.** There is no `if/elif`, no
> `WEATHER_DB`, no return statement anywhere in it. Gemini will never know
> *how* `get_weather` computes its answer, that it's fake/simulated data, or
> that it even exists as Python — only that a capability with this name,
> this description, and this argument shape is *available* to be requested.
>
> **This is also a security boundary worth saying explicitly:** your source
> code, business logic, database credentials embedded in a tool's
> implementation, etc. never leave your machine. Only the interface does.

## 4. Two Ways a Python Function Becomes a "Tool" Declaration

### 4a. Manual — you write the JSON schema by hand
That's what we just did above. You are responsible for keeping the
declaration's `parameters` in sync with the function's real signature — if
you add a parameter to `get_weather` and forget to update the JSON, Gemini
will never know that parameter exists.

### 4b. Automatic — the SDK builds the schema *for* you, by introspection
If you hand the Google GenAI SDK the **raw Python function itself** (not a
JSON dict), it inspects the function the same way you would if you read its
source: it looks at the **parameter names and type hints** via
`inspect.signature`, and the **description** via the docstring. Watch:

In [ ]:
sig = inspect.signature(get_weather)
print("Function name:   ", get_weather.__name__)
print("Parameters:      ", dict(sig.parameters))
print("Return type hint:", sig.return_annotation)
print("Docstring:       ", get_weather.__doc__.strip().splitlines()[0])

# This is conceptually *exactly* what the SDK does internally when you pass
# a raw callable into `tools=[...]` instead of a hand-written declaration --
# it builds the same JSON shape as Section 3, automatically, from the
# function's signature + docstring. You'll see this in action in Section 8.

Function name:    get_weather
Parameters:       {'city': <Parameter "city: str">}
Return type hint: <class 'dict'>
Docstring:        Get the current simulated weather for a city.


| | Manual declaration (3a) | Automatic / introspected (3b) |
|---|---|---|
| Who writes the schema? | You, by hand | The SDK, from your function's signature + docstring |
| Source of the description | Whatever string you type | Your docstring's first line (so **write good docstrings**) |
| Source of "required" params | You list them yourself | Params without a default value |
| Risk of drift | High — easy to forget to update | Low — schema always matches the real signature |
| Used in | The Day 4 agent-loop notebook (full manual control) | Quick scripts, "automatic function calling" (Section 8) |

**Either way, the destination is identical**: a small JSON object containing
a name, a description, and a parameter schema — nothing else ever leaves
your machine.

## 5. How Gemini "Selects" a Tool — There Is No `if` Statement

It's tempting to imagine Gemini has some internal `switch (userIntent)`
statement that routes to a tool. **It doesn't.** Gemini is doing the exact
same thing it always does — predicting the next tokens — except the model has
been trained so that when the *conversation + the list of available tool
schemas* make a function call the most likely continuation, it emits a
structured `function_call` (name + JSON args) instead of prose.

That means **tool selection is a semantic-matching problem, not a logic
problem** — and the single biggest lever you have over it is the
**`description` field**. Watch what happens with the same prompt and two
different tool descriptions.

In [ ]:
def ask(prompt: str, description: str):
    """Send one prompt with ONE tool available, whose description we control."""
    declaration = {
        "name": "get_weather",
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    }
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[declaration])])
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    for part in response.candidates[0].content.parts:
        if part.function_call:
            print(f"  -> requested tool: {part.function_call.name}({dict(part.function_call.args)})")
        if part.text:
            print(f"  -> answered with plain text: {part.text.strip()[:120]}")

prompt = "Is it a good day to hang laundry outside in Tokyo?"

print("Vague description:")
ask(prompt, "Gets information about a city.")

print("\nPrecise description:")
ask(prompt, "Gets the current temperature and weather condition (e.g. rainy, sunny) for a city.")

Vague description:


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 58.304598181s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.6-flash', 'location': 'global'},

Same model, same prompt, same underlying Python function — the only thing
that changed is a sentence of English in the schema. **The description is
the entire "API documentation" Gemini gets to decide *whether* and *when*
your tool is relevant.** A vague description is the #1 real-world cause of
"why didn't it call my tool?" bugs.

### You can also make the "when" explicit yourself — `tool_config`

Selection isn't *purely* left to the model's judgement. You can force the
behaviour with `tool_config`:

| Mode | Behaviour |
|---|---|
| `AUTO` (default) | Gemini decides — tool call or plain text, its choice |
| `ANY` | Gemini **must** call a tool this turn (optionally restricted to a name list) |
| `NONE` | Gemini is **not allowed** to call any tool this turn, even if one would help |

In [ ]:
config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[weather_declaration])],
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",  # force a tool call even though this prompt doesn't obviously need one
            allowed_function_names=["get_weather"],
        )
    ),
)

response = client.models.generate_content(
    model=MODEL,
    contents="Tell me something interesting.",
    config=config,
)
for part in response.candidates[0].content.parts:
    if part.function_call:
        print("Forced call:", part.function_call.name, dict(part.function_call.args))

## 6. The Complete Call Chain — Who Calls Whom, Where

This is the table to put on a slide. Every row is one arrow from the Section
1 diagram, made explicit.

| Step | Who / what does this | Runs on | What happens |
|---|---|---|---|
| 1 | **You** | Your machine | Write `tools=[...]` (schema) and call `client.models.generate_content(...)` |
| 2 | google-genai SDK | Your machine | Serializes your prompt + schema into an HTTPS JSON request |
| 3 | Gemini API | Google's servers | Receives the request; the model reads prompt + schema as context |
| 4 | **Gemini (the model)** | Google's servers | Predicts a structured `function_call` (name + JSON args) instead of prose — **no code executes here** |
| 5 | google-genai SDK | Your machine | Deserializes the HTTP response into a Python `GenerateContentResponse` object |
| 6 | **Your code** | Your machine | Reads `part.function_call.name` and `part.function_call.args` |
| 7 | **Your code** | Your machine | Looks up that name string in *your own* dispatch table (e.g. `TOOLBOX[name]`) |
| 8 | **Your code** | Your machine | Calls the *real* function: `TOOLBOX[name](**args)` — 🔴 **the only step where actual code executes anywhere in this chain** |
| 9 | **Your code** | Your machine | Wraps the return value in a `Part.from_function_response(...)` and sends a *new* request |
| 10 | Gemini (the model) | Google's servers | Reads the result as context, predicts the final answer text |

Notice rows 6–9: **all four of those steps are your own Python code.**
Nothing about them is "AI" — they're a dictionary lookup and a function
call, the same as any plugin system you'd have written in 2015. The only
genuinely new ingredient is row 4: a model that can decide, from plain
English, *when* to ask for one.

## 7. Proving It: Two Experiments

### Experiment A — Gemini "asks" for a tool that doesn't exist on our side
Because Gemini only ever emits *text describing a request*, nothing stops it
from naming a tool your dispatch table doesn't actually have (a typo'd
schema, a hallucinated name, a tool you removed). If that happens, the
failure is entirely **on your side**, at step 7 above — Gemini's servers are
long finished and don't know or care what you do with the request.

In [ ]:
TOOLBOX = {"get_weather": get_weather}  # deliberately does NOT contain "get_forecast"

def execute_tool(name: str, args: dict):
    """Step 7 + 8 from the table above, isolated as one function."""
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(
            f"Gemini requested '{name}', but our TOOLBOX doesn't have it. "
            f"This is OUR error, raised on OUR machine -- Gemini's job ended "
            f"the moment it sent the request."
        )
    return func(**args)

# Simulate Gemini having requested a tool we never registered.
try:
    execute_tool("get_forecast", {"city": "Tokyo"})
except ValueError as e:
    print("Caught locally:", e)

### Experiment B — watch steps 4 and 8 happen for real, one at a time
No hidden loop this time — we'll print a message *between* "Gemini asked for
the tool" and "we actually ran it", so the two moments are visibly separate
in the output.

In [ ]:
config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[weather_declaration])])

response = client.models.generate_content(
    model=MODEL,
    contents="What's the weather in Mumbai right now?",
    config=config,
)

part = response.candidates[0].content.parts[0]
print("STEP 4 (Google's servers, just finished): Gemini requested ->",
      part.function_call.name, dict(part.function_call.args))

print("... nothing has executed yet. Gemini is done. We are now in OUR code ...")

result = execute_tool(part.function_call.name, dict(part.function_call.args))
print("STEP 8 (your machine, happening now):     ran the real function ->", result)

## 8. Automatic Function Calling — Letting the SDK Do Steps 6–9 For You

Everything above was **manual mode**: you wrote the declaration, you parsed
`function_call`, you dispatched it yourself. That's what the agent-loop
notebook needed, because it wanted full control over every step (retries,
logging, a `max_steps` guard, an allow-list).

But if you pass the **raw Python function itself** into `tools=[...]`
(instead of a declaration dict), the SDK will:

1. Build the schema for you by introspection (Section 4b), **and**
2. Automatically run steps 6–9 internally, in a hidden loop, and hand you
   only the *final* text.

Same underlying chain from Section 6 — just steps 6–9 happen inside the SDK
instead of inside your own loop.

In [ ]:
def get_current_time_str(timezone: str) -> str:
    """Get the current time for an IANA timezone.

    Args:
        timezone: IANA timezone name, e.g. "Asia/Tokyo".
    """
    from datetime import datetime
    from zoneinfo import ZoneInfo
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y at %I:%M:%S %p")

# Pass the RAW FUNCTIONS -- not declarations, not a TOOLBOX -- and let the
# SDK introspect + dispatch automatically.
auto_config = types.GenerateContentConfig(tools=[get_weather, get_current_time_str])

response = client.models.generate_content(
    model=MODEL,
    contents="What's the weather in Paris, and what time is it there?",
    config=auto_config,
)

print("FINAL TEXT (the only thing you had to ask for):")
print(response.text)

# The SDK still went through every row of the Section 6 table internally --
# this history proves it, showing the exact function_call / function_response
# turns it generated and dispatched on your behalf.
history = getattr(response, "automatic_function_calling_history", None)
if history:
    print("\nHidden steps the SDK ran for you:")
    for turn in history:
        for p in turn.parts:
            if p.function_call:
                print("  requested:", p.function_call.name, dict(p.function_call.args))
            if p.function_response:
                print("  executed, returned:", p.function_response.response)

| | Manual mode (Sections 3–7) | Automatic function calling (this section) |
|---|---|---|
| Who writes the schema? | You (or you rely on 4b's introspection yourself) | The SDK, automatically |
| Who runs steps 6–9? | Your own loop / your own `TOOLBOX` | The SDK, internally |
| Can you add a `max_steps` guard, logging, an approval gate? | Yes — it's your loop | Not without opting back into manual mode |
| Best for | Production agents, anything with side effects (Day 4 agent notebook) | Quick scripts, prototypes, read-only tools |

**Rule of thumb for your own projects:** if a tool can *change* something
(send an email, write a row, spend money), prefer manual mode — you want a
visible line of code (step 7/8) where you can insert a guardrail. Automatic
mode is for convenience, not for anything you wouldn't want to run
unsupervised.

## 9. 🎤 Crystal-Clear Cheat Sheet — Read This Straight to the Class

| Question | Answer |
|---|---|
| Does Gemini run my Python code? | **No.** It only ever generates text. |
| Then who runs it? | **Your own process**, at step 8 of the Section 6 table. |
| What does Gemini receive about my function? | Only its **name, description, and parameter schema** — never the source code. |
| How does Gemini "choose" a tool? | The same way it predicts any token: by conditioning on the conversation **and the schema text** — the `description` field matters most. |
| Can I control *when* it's allowed to call a tool? | Yes — `tool_config` with mode `AUTO` / `ANY` / `NONE`. |
| What if Gemini requests a tool I never defined? | Your dispatch code raises the error, locally. Gemini's servers never find out. |
| Do I have to hand-write the JSON schema every time? | No — pass the raw function into `tools=[...]` and the SDK introspects it for you. |
| Who decides how many tool calls happen in a row? | In manual mode, **your loop** (see the agent-loop notebook). In automatic mode, the SDK's hidden loop. |

## 10. 🧪 Classroom Challenge

For the prompt **"Convert today's temperature in Tokyo to Fahrenheit."**,
fill in this table *before* running anything — then verify with code:

| Step # (from Section 6's table) | Who does it | What exactly happens for this prompt? |
|---|---|---|
| 1 | ? | ? |
| 4 | ? | ? |
| 8 | ? | ? |
| 9 | ? | ? |
| 10 | ? | ? |

**Bonus question for discussion:** this prompt needs *two* tool calls
(`get_weather`, then a conversion). At which exact row of the Section 6
table does the *second* request for a tool get generated — and why can't
Gemini ask for both tools in a single row-4 moment?

*(Answer: it can request both in the same turn if it doesn't need the first
result to compute the second's arguments — but here it does, so row 4 for the
conversion can only happen after a fresh row 3, once the weather result has
gone back in as context. That's exactly why the agent-loop notebook needed a
loop instead of one round trip.)*

## 🎓 Day 4 Takeaway

By the end of this notebook, you should be able to explain, without
hesitating:

1. **Gemini never executes your code** — it only ever generates text that
   describes a request (a function name + JSON arguments).
2. **Only the interface crosses the network** — name, description,
   parameter schema. Never the function body.
3. A declaration can be **hand-written** (full control) or **auto-generated
   by introspection** from a raw Python function (less boilerplate).
4. "Tool selection" is **semantic matching against schema text**, driven
   overwhelmingly by the `description` field — not a hardcoded router.
   `tool_config` (`AUTO`/`ANY`/`NONE`) lets you steer that decision.
5. The actual function call happens in **your own process**, at one
   specific, inspectable line of your own code — whether you write that
   line yourself (manual mode) or the SDK writes it for you (automatic
   function calling).
6. Manual mode exists precisely so you can put a **guardrail** at that one
   line — which is why the agent-loop notebook builds its own loop instead
   of relying on automatic function calling.

### The one diagram to remember

```text
 YOUR MACHINE                                    GOOGLE'S SERVERS
 ─────────────                                    ─────────────────
 real Python function  ──(name + schema only)──▶  Gemini reads + decides
                                                          │
 YOUR code runs the      ◀──(text: name + args)──────────┘
 real function here
      │
      └──(result)──────────────────────────────▶  Gemini reads + answers
```

## Official references

- Gemini API — Function calling: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API — Automatic function calling (Python SDK): https://ai.google.dev/gemini-api/docs/function-calling#automatic_function_calling_python_only
- Gemini API — Tools overview: https://ai.google.dev/gemini-api/docs/tools
- Gemini API — Getting started: https://ai.google.dev/gemini-api/docs/get-started
- Python `inspect` module (what schema introspection is built on): https://docs.python.org/3/library/inspect.html

See also: [1-Building_AI_Agents.ipynb](./1-Building_AI_Agents.ipynb) for the
full agent loop built on top of everything in this notebook, and
[1-MCP_Server_Basics.ipynb](./1-MCP_Server_Basics.ipynb) for tools you don't
have to write the schema for at all.